In [ ]:
import torch
import math
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dim = 2
size = 256

class PeriodicGRF:
    def __init__(self, dim, size, kernel="matern", device="cpu", **kernel_params):
        """
        dim: 1 or 2
        size: grid size per dim
        kernel: kernel name string
        kernel_params: kernel-specific params
        """
        self.dim = dim
        self.size = size
        self.device = device
        self.kernel = kernel.lower()
        self.params = kernel_params

        freqs_1d = torch.fft.fftfreq(size, d=1.0/size).to(device)

        if dim == 1:
            self.k = freqs_1d
        elif dim == 2:
            kx, ky = torch.meshgrid(freqs_1d, freqs_1d, indexing='ij')
            self.k = torch.sqrt(kx**2 + ky**2)
        else:
            raise NotImplementedError("Only dim=1 or 2 supported.")

        self.sqrt_spec_density = self._compute_sqrt_spectral_density()

    def _compute_sqrt_spectral_density(self):
        k = self.k
        p = self.params
        kernel = self.kernel

        if kernel == "matern":
            # Matern: (4pi^2 k^2 + tau^2)^(-alpha/2)
            alpha = p.get("alpha", 2.5)
            tau = p.get("tau", 7.0)
            sigma = p.get("sigma", tau ** (0.5*(2*alpha - dim)))
            spec = (4 * (math.pi ** 2) * k ** 2 + tau ** 2) ** (-alpha / 2)
            spec[0] = 0.0
            spec = (self.size**self.dim) * math.sqrt(2.0) *  sigma * spec

        elif kernel == "rbf" or kernel == "squared_exponential":
            length_scale = p.get("length_scale", 0.1)
            variance = p.get("variance", 1.0)
            spec = variance * torch.exp(-2 * (math.pi ** 2) * (length_scale ** 2) * k ** 2)
            spec[0] = 0.0

        elif kernel == "exponential":
            length_scale = p.get("length_scale", 0.1)
            variance = p.get("variance", 1.0)
            spec = variance / (1 + (2 * math.pi * length_scale * k) ** 2)
            spec[0] = 0.0

        elif kernel == "rq" or kernel == "rational_quadratic":
            length_scale = p.get("length_scale", 0.1)
            alpha = p.get("alpha", 1.0)
            variance = p.get("variance", 1.0)
            spec = variance * (1 + (k ** 2) / (2 * alpha * length_scale ** 2)) ** (-alpha)
            spec[0] = 0.0

        elif kernel == "periodic":
            # Periodic kernel谱是离散峰，近似实现用多个谱线叠加，简单用正弦谱叠加示例
            # 这里示范用单频率谱峰近似周期性（只能近似）
            length_scale = p.get("length_scale", 0.1)
            variance = p.get("variance", 1.0)
            freq0 = p.get("freq", 1.0)
            # 频率中心峰位置
            spec = torch.zeros_like(k)
            peak_width = 0.05  # 峰宽参数可调
            spec += variance * torch.exp(-((k - freq0) ** 2) / (2 * peak_width ** 2))
            spec += variance * torch.exp(-((k + freq0) ** 2) / (2 * peak_width ** 2))
            spec[0] = 0.0

        elif kernel == "linear":
            # 线性核非平稳，频谱不存在简单闭式，暂不支持
            raise NotImplementedError("Linear kernel not supported in spectral form.")

        elif kernel == "spectral_mixture":
            # 频谱为多个高斯峰叠加
            # params: weights, means, variances — 都是列表或tensor
            weights = torch.tensor(p.get("weights", [1.0]), device=self.device)
            means = torch.tensor(p.get("means", [0.0]), device=self.device)
            variances = torch.tensor(p.get("variances", [0.1]), device=self.device)
            spec = torch.zeros_like(k)
            for w, m, v in zip(weights, means, variances):
                spec += w * torch.exp(-0.5 * ((k - m) ** 2) / v)
            spec[0] = 0.0

        elif kernel == "cauchy":
            # Cauchy kernel ~ (1 + (k/l)^2)^(-alpha)
            length_scale = p.get("length_scale", 0.1)
            alpha = p.get("alpha", 1.0)
            variance = p.get("variance", 1.0)
            spec = variance * (1 + (k / length_scale) ** 2) ** (-alpha)
            spec[0] = 0.0

        elif kernel == "powered_exponential":
            # 广义指数核，spec ~ exp(-(k*l)^gamma), gamma in (0,2]
            length_scale = p.get("length_scale", 0.1)
            gamma = p.get("gamma", 1.5)
            variance = p.get("variance", 1.0)
            spec = variance * torch.exp(-(k * length_scale) ** gamma)
            spec[0] = 0.0

        elif kernel == "wave":
            # 波动核示意，谱为双峰对应振荡频率
            freq0 = p.get("freq", 5.0)
            variance = p.get("variance", 1.0)
            spec = variance * (torch.exp(-0.5 * ((k - freq0) ** 2) / 0.1) +
                               torch.exp(-0.5 * ((k + freq0) ** 2) / 0.1))
            spec[0] = 0.0

        else:
            raise ValueError(f"Unknown kernel: {kernel}")

        # return torch.sqrt(spec)
        return spec

    def sample(self, N):
        shape = (N,) + self.k.shape
        coeff = torch.randn(shape, dtype=torch.cfloat, device=self.device)
        coeff = coeff * self.sqrt_spec_density
        if self.dim == 1:
            field = torch.fft.ifft(coeff, dim=-1).real
        else:
            field = torch.fft.ifftn(coeff, dim=(-2, -1)).real
        return field
    
def time_func(func, *args, **kwargs):
    start = time.time()
    result = func(*args, **kwargs)
    end = time.time()
    print(f"{func.__name__} took {end - start:.4f} seconds")
    return result

def generate_matern(alpha,tau,N):
    grf_matern = PeriodicGRF(dim, size, kernel="matern", device=device, alpha=alpha, tau=tau)
    samples = grf_matern.sample(N=N)
    return samples

def generate_student_t(chi2,df,sample_grf):
    return sample_grf / (torch.sqrt(chi2 / df).view(-1,1,1))

def format_params(name, params):
    param_str = ", ".join(f"{k}={v}" for k, v in params.items())
    return f"{name}\n({param_str})"

def visualize_fields_with_params(fields_list, titles_with_params, ncols=3, figsize=(15,10), cmap="viridis"):
    """
    fields_list: list of tensors, each shape (N, H, W) or (N, size,...)
    titles_with_params: list of str, each corresponding to fields_list element (including kernel name & params)
    ncols: number of columns in the plot grid
    """
    nplots = len(fields_list)
    nrows = (nplots + ncols - 1) // ncols

    fig, axs = plt.subplots(nrows, ncols, figsize=figsize)
    axs = axs.flatten()

    for i, (field, title) in enumerate(zip(fields_list, titles_with_params)):
        ax = axs[i]
        # 只画第一个样本（index=0），二维张量
        img = ax.imshow(field[0].cpu(), cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)

    # 剩余空白子图隐藏
    for j in range(i+1, len(axs)):
        axs[j].axis("off")

    plt.tight_layout()
    plt.show()

def plot_matern_heatmaps(alpha,tau,exp_factor,df):
    N = 1

    sample_grf = time_func(generate_matern, alpha, tau, N)

    sample_log_grf = torch.exp(exp_factor*sample_grf)
    mean_per_sample = sample_log_grf.mean(dim=(1,2), keepdim=True)  # shape (10,1,1)
    sample_log_grf = sample_log_grf - mean_per_sample

    chi2 = torch.distributions.Chi2(df).sample((N,)).to(device)
    student_t_field = time_func(generate_student_t, chi2, df, sample_grf)


    # 把你生成的结果放入列表和标题
    fields = [
        sample_grf,          # Matern GRF
        sample_log_grf, # exp(GRF)
        student_t_field,     # Student-t field
    ]

    titles_with_params = [
        format_params("Matern GRF", {"kernel": "matern", "alpha": alpha, "tau": tau}),
        format_params("Exp Matern GRF", {"kernel": "matern", "alpha": alpha, "tau": tau, "exp_factor": exp_factor}),
        format_params("Student-t field Matern GRF", {"df": df}),
    ]

    visualize_fields_with_params(fields, titles_with_params, ncols=3, figsize=(15,10))

In [ ]:
for alpha in [1.5,2,2.5,3,3.5]:
    for tau in [1,3,5,7,9]:
        for exp_factor in [0.5,1,1.5,2]:
            for df in [1,2]:
                print("alpha: ",alpha,", tau: ",tau,", exp_factor: ",exp_factor,", df: ",df)
                plot_matern_heatmaps(alpha,tau,exp_factor,df)

alpha:  0.5 , tau:  0 , exp_factor:  0.5 , df:  1


ZeroDivisionError: 0.0 cannot be raised to a negative power